# Task B: 3D U-Net for Catheter Segmentation

(using the same architecture as Task A but trained on catheter GT masks instead)

- labels: `subjXXX_catheter.nii.gz` (values 0/1/2 → binarized to 0/1)
- Post-processing: catheter predictions that overlap with the Task A artifact mask are removed as a constraint

In [ ]:
%%capture
!pip install monai nibabel tqdm matplotlib

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not running in Colab.')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('WARNING: No GPU detected. Enable GPU: Runtime > Change runtime type > T4 GPU.')

In [ ]:
import json
import random
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader

from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRangePercentilesd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandShiftIntensityd,
    Lambdad,
)
from monai.data import CacheDataset, list_data_collate

print('Imports OK.')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
if IN_COLAB:
    DATA_ROOT = Path('/content/drive/MyDrive/Project 2')
else:
    DATA_ROOT = Path('/Volumes/LUCY DISK/mia/Project 2')

GT_DIR    = DATA_ROOT / 'gt_dir'
PRED_DIR  = DATA_ROOT / 'pred_dir'
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = DATA_ROOT / 'unet_taskB_best.pth'   # on Drive so it survives runtime crashes

# ── Hyperparameters ────────────────────────────────────────────────────────
PATCH_SIZE   = (64, 64, 64)
NUM_SAMPLES  = 4
BATCH_SIZE   = 4
LR           = 2e-4
MAX_EPOCHS   = 40
VAL_INTERVAL = 5
VAL_FRAC     = 0.2
THRESHOLD    = 0.5

PIN_MEMORY = device.type == 'cuda'

print('GT dir exists:', GT_DIR.exists())
print('Pred dir:', PRED_DIR)

In [ ]:
def is_real_nifti(path: Path) -> bool:
    return (
        path.exists()
        and not path.name.startswith('._')
        and path.stat().st_size > 1_000
    )

subject_dirs = sorted(
    p for p in GT_DIR.iterdir()
    if p.is_dir() and p.name.startswith('subj') and not p.name.startswith('._')
)

data_dicts      = []
skipped_sidecar = []
skipped_missing = []

for sd in subject_dirs:
    s = sd.name
    img_p = sd / f'{s}_image.nii.gz'
    lbl_p = sd / f'{s}_catheter.nii.gz'   # Task B label

    if is_real_nifti(img_p) and is_real_nifti(lbl_p):
        data_dicts.append({'image': str(img_p), 'label': str(lbl_p), 'subject': s})
    else:
        has_sidecar = (sd / f'._{s}_image.nii.gz').exists() or (sd / f'._{s}_catheter.nii.gz').exists()
        if has_sidecar:
            skipped_sidecar.append(s)
        else:
            skipped_missing.append(s)

print(f'Usable subjects: {len(data_dicts)}')
if skipped_sidecar:
    print(f'Skipped — macOS ._ sidecar only ({len(skipped_sidecar)}): {skipped_sidecar[:8]}')
if skipped_missing:
    print(f'Skipped — files absent ({len(skipped_missing)}): {skipped_missing[:8]}')

assert len(data_dicts) >= 2, 'Need at least 2 subjects.'

random.seed(42)
random.shuffle(data_dicts)
n_val       = max(1, int(len(data_dicts) * VAL_FRAC))
train_dicts = data_dicts[n_val:]
val_dicts   = data_dicts[:n_val]
print(f'Train: {len(train_dicts)}  |  Val: {len(val_dicts)}')

In [ ]:
# The catheter mask has values [0, 1, 2] — binarise to 0/1 with a Lambdad transform.
binarise = Lambdad(keys=['label'], func=lambda x: (x > 0).float())

train_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    ScaleIntensityRangePercentilesd(
        keys=['image'], lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True,
    ),
    binarise,
    RandCropByPosNegLabeld(
        keys=['image', 'label'],
        label_key='label',
        spatial_size=PATCH_SIZE,
        pos=1, neg=1,
        num_samples=NUM_SAMPLES,
        image_key='image',
        image_threshold=0,
    ),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
    RandShiftIntensityd(keys=['image'], offsets=0.1, prob=0.3),
])

val_transforms = Compose([
    LoadImaged(keys=['image', 'label']),
    EnsureChannelFirstd(keys=['image', 'label']),
    ScaleIntensityRangePercentilesd(
        keys=['image'], lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True,
    ),
    binarise,
])

train_ds = CacheDataset(data=train_dicts, transform=train_transforms, cache_rate=0.25, num_workers=0)
val_ds   = CacheDataset(data=val_dicts,   transform=val_transforms,   cache_rate=1.0,  num_workers=0)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=list_data_collate, num_workers=0, pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print(f'Train batches/epoch: {len(train_loader)}  (effective patches/batch: {BATCH_SIZE * NUM_SAMPLES})')
print(f'Val subjects: {len(val_loader)}')

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
).to(device)

loss_fn   = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2 * inter / denom)


FORCE_RETRAIN = False

if CKPT_PATH.exists() and not FORCE_RETRAIN:
    print(f'Checkpoint found at {CKPT_PATH} — skipping training.')
    print('Set FORCE_RETRAIN = True to train from scratch.')
else:
    best_dice   = 0.0
    train_losses, val_dices, val_epoch_pts = [], [], []
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

    epoch_bar = tqdm(range(1, MAX_EPOCHS + 1), desc='Epochs', unit='ep')
    for epoch in epoch_bar:
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        batch_bar = tqdm(train_loader, desc=f'  Ep {epoch:3d} train', leave=False, unit='batch')
        for batch in batch_bar:
            imgs   = batch['image'].to(device)
            labels = batch['label'].long().to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                logits = model(imgs)
                loss   = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            batch_bar.set_postfix(loss=f'{loss.item():.4f}')
        epoch_loss /= len(train_loader)
        scheduler.step()
        train_losses.append(epoch_loss)

        # ── Validate ───────────────────────────────────────────────────────
        if epoch % VAL_INTERVAL == 0 or epoch == MAX_EPOCHS:
            model.eval()
            subj_dices = []
            with torch.no_grad():
                for vdata in tqdm(val_loader, desc=f'  Ep {epoch:3d} val  ', leave=False, unit='subj'):
                    v_img = vdata['image'].to(device)
                    v_lbl = vdata['label'].cpu().numpy()[0, 0]
                    with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                        v_logits = sliding_window_inference(
                            v_img, PATCH_SIZE, sw_batch_size=4,
                            predictor=model, overlap=0.25,
                        )
                    v_proba = torch.softmax(v_logits.float(), dim=1)[0, 1].cpu().numpy()
                    subj_dices.append(dice_score(v_proba > THRESHOLD, v_lbl > 0))
            mean_dice = float(np.mean(subj_dices))
            val_dices.append(mean_dice)
            val_epoch_pts.append(epoch)
            ckpt_marker = ''
            if mean_dice > best_dice:
                best_dice = mean_dice
                torch.save(model.state_dict(), CKPT_PATH)
                ckpt_marker = ' ✓ saved'
            epoch_bar.write(
                f'Epoch {epoch:3d}/{MAX_EPOCHS} | loss={epoch_loss:.4f} | val_dice={mean_dice:.4f}{ckpt_marker}'
            )

    print(f'\nTraining complete. Best val Dice = {best_dice:.4f}')

In [ ]:
try:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(range(1, len(train_losses) + 1), train_losses)
    ax1.set(xlabel='Epoch', ylabel='DiceCE Loss', title='Training Loss')
    ax2.plot(val_epoch_pts, val_dices, marker='o', color='tab:orange')
    ax2.set(xlabel='Epoch', ylabel='Dice (catheter)', title='Validation Dice')
    plt.tight_layout()
    plt.show()
except NameError:
    print('No training history to plot (training was skipped).')

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()
print(f'Loaded checkpoint: {CKPT_PATH}')

In [ ]:
from scipy import ndimage as ndi

infer_transforms = Compose([
    LoadImaged(keys=['image']),
    EnsureChannelFirstd(keys=['image']),
    ScaleIntensityRangePercentilesd(
        keys=['image'], lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True,
    ),
])

SUBJECTS = [f'subj{i:03d}' for i in range(1, 31)]

for subj in tqdm(SUBJECTS):
    img_path = GT_DIR / subj / f'{subj}_image.nii.gz'
    if not img_path.exists():
        print(f'[SKIP] {subj}: image not found')
        continue

    data    = infer_transforms({'image': str(img_path)})
    img_t   = data['image'].unsqueeze(0).to(device)
    ref_nii = nib.load(str(img_path))

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            logits = sliding_window_inference(
                img_t, PATCH_SIZE, sw_batch_size=4, predictor=model, overlap=0.25,
            )
    proba     = torch.softmax(logits.float(), dim=1)[0, 1].cpu().numpy()
    pred_mask = (proba > THRESHOLD).astype(np.uint8)

    # Keep only the largest connected component (catheter is one continuous structure)
    labeled, n = ndi.label(pred_mask)
    if n > 1:
        sizes = ndi.sum(pred_mask, labeled, range(1, n + 1))
        pred_mask = (labeled == (np.argmax(sizes) + 1)).astype(np.uint8)

    # Remove any voxels overlapping the Task A artifact prediction
    taskA_path = PRED_DIR / f'{subj}_taskA.nii.gz'
    if taskA_path.exists():
        artifact_mask = nib.load(str(taskA_path)).get_fdata().astype(bool)
        pred_mask[artifact_mask] = 0

    out_nii  = nib.Nifti1Image(pred_mask, affine=ref_nii.affine, header=ref_nii.header)
    nib.save(out_nii, str(PRED_DIR / f'{subj}_taskB.nii.gz'))

print(f'Task B predictions saved to: {PRED_DIR}')

In [ ]:
# Evaluate Task B predictions against GT where available.
dices = []
for subj in SUBJECTS:
    pred_path = PRED_DIR / f'{subj}_taskB.nii.gz'
    gt_path   = GT_DIR / subj / f'{subj}_catheter.nii.gz'
    if not pred_path.exists() or not gt_path.exists():
        continue
    pred = nib.load(str(pred_path)).get_fdata()
    gt   = (nib.load(str(gt_path)).get_fdata() > 0).astype(np.uint8)
    d = dice_score(pred, gt)
    dices.append(d)
    print(f'{subj}: Dice={d:.4f}  pred_vox={int(pred.sum())}  gt_vox={int(gt.sum())}')

if dices:
    print(f'\nMean Dice over {len(dices)} subjects: {np.mean(dices):.4f}')